# 05. Quality Diversity: MAP-Elites and Differentiable QD

- **Category:** 09_causal_ai_and_evolutionary_search
- **Lineage & Citations:** Lehman & Stanley (2011 *Novelty Search*); Mouret & Clune (2015 *Illuminating search spaces by mapping elites*); Fontaine & Nikolaidis (2021 *Differentiable Quality Diversity*).
- **Purpose:** Illuminate high-dimensional parameter spaces across diverse behavioral niches rather than prematurely converging to a single local optimum.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Evolution Strategies CMA-ES and PSO](04_evolution_strategies_cma_es_openai_es_and_pso.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. Population Based Training and DARTS](06_population_based_training_pbt_and_darts_neural_architecture_search.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use rectilinear MAP-Elites grids when descriptor dimensionality $K > 6$ (curse of dimensionality causing exponential cell explosion $\prod B_k$); use CVT-MAP-Elites or single-objective optimizers for strictly convex unconstrained objectives.

## Mathematical Foundations & Governing Equations

1. **Continuous Descriptor Binning**: $\text{idx}_k = \text{clamp}\left(\left\lfloor \frac{b_k - b_{\min, k}}{b_{\max, k} - b_{\min, k}} B_k \right\rfloor, 0, B_k - 1\right), \quad c = \text{idx}_0 B_1 + \text{idx}_1$
2. **Elite Replacement**: $\mathcal{A}[c] \leftarrow \max(\mathcal{A}[c], f_i)$
3. **DQD Gradient Arborescence**: $\Delta \theta = c_0 \nabla_\theta f + \sum_k c_k \nabla_\theta b_k$
4. **Orthogonalized Diversity Stepping**: $g_k^\perp = g_k - \frac{\langle g_k, g_0 \rangle}{\|g_0\|^2 + \epsilon} g_0$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/09_causal_ai_and_evolutionary_search/05_quality_diversity_map_elites_and_differentiable_qd.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def map_elites_indices(desc, b_min, b_max, grid_size):
    scale = (desc - b_min.unsqueeze(0)) / (b_max.unsqueeze(0) - b_min.unsqueeze(0) + 1e-8)
    indices = torch.clamp(torch.floor(scale * grid_size.unsqueeze(0)).long(), torch.zeros_like(grid_size.long()), (grid_size - 1).long())
    cell_idx = indices[:, 0] * grid_size[1].long() + indices[:, 1]
    return indices.float(), cell_idx.float()

def map_elites_update(arch, cand_fit, cell_idx):
    res = arch.clone()
    for i in range(cand_fit.shape[0]):
        c = int(cell_idx[i].item())
        if cand_fit[i] > res[c]: res[c] = cand_fit[i]
    return res

def dqd_arborescence(grad_f, jac_b, coeffs):
    return coeffs[0] * grad_f + torch.sum(coeffs[1:].unsqueeze(-1) * jac_b, dim=0)

def dqd_ortho_jac(grad_f, jac_b, eps=1e-8):
    norm_sq = torch.sum(grad_f * grad_f) + eps
    return jac_b - ((jac_b @ grad_f) / norm_sq).unsqueeze(-1) * grad_f.unsqueeze(0)

In [ ]:
# Parity verification against golden reference tensors
g_idx, c_idx = map_elites_indices(weights["descriptors"], weights["b_min"], weights["b_max"], weights["grid_size"])
assert torch.max(torch.abs(g_idx - weights["grid_indices_ref"])) < 1e-5, "Grid indices mismatch"
assert torch.max(torch.abs(c_idx - weights["cell_indices_ref"])) < 1e-5, "Cell indices mismatch"

up = map_elites_update(weights["archive_fit"], weights["cand_fitness"], weights["cell_indices_ref"])
assert torch.max(torch.abs(up - weights["archive_fit_updated_ref"])) < 1e-5, "Archive update mismatch"

step = dqd_arborescence(weights["grad_f"], weights["jac_b"], weights["dqd_coeffs"])
assert torch.max(torch.abs(step - weights["dqd_step_ref"])) < 1e-5, "DQD step mismatch"

ortho = dqd_ortho_jac(weights["grad_f"], weights["jac_b"])
assert torch.max(torch.abs(ortho - weights["dqd_ortho_jac_ref"])) < 1e-5, "DQD ortho mismatch"
print("All MAP-Elites and DQD parity checks passed with max error < 1e-5!")

In [ ]:
# Benchmarking MAP-Elites and DQD operators
t0 = time.perf_counter()
for _ in range(500):
    _, c = map_elites_indices(weights["descriptors"], weights["b_min"], weights["b_max"], weights["grid_size"])
    _ = map_elites_update(weights["archive_fit"], weights["cand_fitness"], c)
    _ = dqd_arborescence(weights["grad_f"], weights["jac_b"], weights["dqd_coeffs"])
    _ = dqd_ortho_jac(weights["grad_f"], weights["jac_b"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused MAP-Elites + DQD latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Single-objective hill climbing getting trapped vs MAP-Elites stepping stones
# Deceptive 2D Rastrigin-like landscape with global optimum behind a fitness moat
def fitness_fn(theta): return -(theta[0]**2 + theta[1]**2) + 4.0 * torch.cos(3.0 * theta[0])
# Greedy gradient hill-climber from deceptive starting point
th_greedy = torch.tensor([1.8, 0.0], requires_grad=True)
opt = torch.optim.SGD([th_greedy], lr=0.05)
for _ in range(50):
    loss = -fitness_fn(th_greedy)
    opt.zero_grad(); loss.backward(); opt.step()
f_greedy = fitness_fn(th_greedy).item()
# MAP-Elites illuminates across diverse behavioral descriptor b = [theta_0]
grid = torch.full((10,), -100.0)
for x_val in torch.linspace(-2.5, 2.5, 50):
    cand = torch.tensor([x_val, 0.0])
    f_val = fitness_fn(cand)
    idx = int(torch.clamp(torch.floor((x_val + 2.5) / 5.0 * 10), 0, 9).item())
    if f_val > grid[idx]: grid[idx] = f_val
f_map_max = torch.max(grid).item()
print(f"Greedy Local Trap Fitness: {f_greedy:.2f} vs MAP-Elites Global Peak: {f_map_max:.2f}")
assert f_map_max > f_greedy + 1.0, "MAP-Elites must escape the deceptive local optimum!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **MAP-Elites** illuminates continuous behavioral feature spaces, maintaining top-performing solutions per niche.
2. **Differentiable QD** combines objective gradients $\nabla_\theta f$ and behavioral Jacobians $J_b$ to guide exploration with first-order efficiency.
3. **Orthogonalization** allows discovering novel behavioral features without sacrificing task performance.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_quality_diversity_map_elites_and_differentiable_qd.md](../../markdown/09_causal_ai_and_evolutionary_search/05_quality_diversity_map_elites_and_differentiable_qd.md)
- **PyTorch 2.x Module:** [05_quality_diversity_map_elites_and_differentiable_qd.py](../../pytorch2/09_causal_ai_and_evolutionary_search/05_quality_diversity_map_elites_and_differentiable_qd.py)
- **Functional JAX Module:** [05_quality_diversity_map_elites_and_differentiable_qd.py](../../jax/09_causal_ai_and_evolutionary_search/05_quality_diversity_map_elites_and_differentiable_qd.py)
- **Native Rust Candle Track:** [05_quality_diversity_map_elites_and_differentiable_qd.rs](../../candle/09_causal_ai_and_evolutionary_search/05_quality_diversity_map_elites_and_differentiable_qd.rs)
- **Hardware Kernel (CPU / WGSL):** [05_quality_diversity_map_elites_and_differentiable_qd.rs](../../kernels/09_causal_ai_and_evolutionary_search/05_quality_diversity_map_elites_and_differentiable_qd.rs)
- **Golden SafeTensors:** [05_quality_diversity_map_elites_and_differentiable_qd.safetensors](../../weights/09_causal_ai_and_evolutionary_search/05_quality_diversity_map_elites_and_differentiable_qd.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Evolution Strategies CMA Es Openai Es and Pso](04_evolution_strategies_cma_es_openai_es_and_pso.ipynb) | [Master Index](../../README.md#09_causal_ai_and_evolutionary_search) | [Next: 06. Population Based Training PBT and DARTS Neural Architecture Search](06_population_based_training_pbt_and_darts_neural_architecture_search.ipynb) &rarr;